#Approach
Get movies rated by user
Multiply movie vector by rating
Sum all vectors
Normalize by total rating

In [1]:
import numpy as np   # used for numerical operations on vectors

In [2]:
# Import PySpark session to work with big data
from pyspark.sql import SparkSession  # used to start spark environment

# Import pandas for local dataframe operations
import pandas as pd  # used for easier processing for TF-IDF

# Import TF-IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer  # converts text into TF-IDF vectors

# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity  # computes similarity between vectors

In [3]:
spark = SparkSession.builder.appName("MovieLensRecommender").getOrCreate()  # start spark session

26/03/14 18:36:05 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/14 18:36:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 18:36:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Path to your dataset
data_path = "/home/rajesh/CSL7100/Assignment3/ml-latest-small/"

ratings_spark = spark.read.csv(data_path + "ratings.csv", header=True, inferSchema=True)  
# load ratings dataset using spark

ratings_spark.show(5)  # preview ratings data

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows



In [5]:
ratings = ratings_spark.toPandas()  
# convert spark dataframe to pandas for easier manipulation

In [7]:
# Load movies dataset using Spark
movies_spark = spark.read.csv(data_path + "movies.csv", header=True, inferSchema=True)  
# header=True reads column names, inferSchema automatically detects column types

movies_spark.show(5)  # preview dataset

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



In [8]:
movies = movies_spark.toPandas()  
# convert Spark dataframe to pandas because sklearn works with pandas/numpy

In [13]:
movieId_to_index = pd.Series(movies.index, index=movies["movieId"]).to_dict()  
# create dictionary mapping movieId → tfidf row index


movies["genres"] = movies["genres"].str.replace("|", " ")  
# replace "|" with space so TF-IDF treats each genre as a word


tfidf = TfidfVectorizer(stop_words="english")  
# create TF-IDF vectorizer; removes common words if present

tfidf_matrix = tfidf.fit_transform(movies["genres"])  
# convert movie genres into TF-IDF feature vectors



In [14]:
def build_user_profile(user_id):

    user_ratings = ratings[ratings["userId"] == user_id]  
    # select all movies rated by this user
    
    weighted_vectors = []  
    # list to store rating-weighted movie vectors
    
    rating_sum = user_ratings["rating"].sum()  
    # total rating sum for normalization
    
    for _, row in user_ratings.iterrows():
        
        movie_id = row["movieId"]  
        rating = row["rating"]
        
        if movie_id in movieId_to_index:
            
            idx = movieId_to_index[movie_id]  
            # get tfidf row index for this movie
            
            movie_vector = tfidf_matrix[idx].toarray()[0]  
            # extract movie TF-IDF vector
            
            weighted_vectors.append(movie_vector * rating)  
            # multiply vector by rating weight
    
    if len(weighted_vectors) == 0:
        return None
    
    user_profile = np.sum(weighted_vectors, axis=0) / rating_sum  
    # compute weighted average vector
    
    return user_profile

In [37]:
def recommend_movies_user(user_id, top_n=5):
    
    user_profile = build_user_profile(user_id)  
    # generate user preference vector

    print("user_profile.shape: ", user_profile.shape)
    #print(user_profile)
    
    if user_profile is None:
        print("No ratings available for this user")
        return

    print("tfidf_matrix.shape: ", tfidf_matrix.shape)
    similarity_scores = cosine_similarity([user_profile], tfidf_matrix)
    # compute similarity between user profile and all movies
    print("similarity_scores.shape: ", similarity_scores.shape)

    similarity_scores = similarity_scores[0]  #2D to 1D
    
    user_rated_movies = ratings[ratings["userId"] == user_id]["movieId"].tolist()  
    # get movies already rated by user
    print("len(user_rated_movies): ", len(user_rated_movies))
    
    scores = list(enumerate(similarity_scores))  
    # attach movie indices to similarity scores
    print("len(scores): ", len(scores))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)  
    # sort movies by similarity
    
    recommendations = []
    
    for idx, score in scores:                     # scores is tuple of (movie_index, similarity_score)
        movie_id = movies.iloc[idx]["movieId"]
        
        if movie_id not in user_rated_movies:  
            # skip movies already watched. We need to give recommendation for the movies which the user has not watched.
            recommendations.append((movies.iloc[idx]["title"], score))
        
        if len(recommendations) == top_n:
            break
    
    return pd.DataFrame(recommendations, columns=["Movie", "SimilarityScore"])

In [38]:
recommend_movies_user(user_id=1, top_n=5)

user_profile.shape:  (23,)
tfidf_matrix.shape:  (9742, 23)
similarity_scores.shape:  (1, 9742)
len(user_rated_movies):  232
len(scores):  9742


,Movie,SimilarityScore
0,Dragonheart 2: A New Beginning (2000),0.791026
1,"Hunting Party, The (2007)",0.773645
2,Flashback (1990),0.773089
3,The Great Train Robbery (1978),0.773089
4,Charlie's Angels: Full Throttle (2003),0.760503


In [ ]:
spark.stop()